In [ ]:
import pandas as pd
import io
import re
from google.colab import files

def processar_relatorio_listagem():
    uploaded = files.upload()

    # Colunas atualizadas: 'Base de Custo' no início e remoção das vigências
    colunas_modelo = [
        'Base de Custo', 'Cia', 'Cardapio', 'Resumo_Cardapio',
        'Class', 'Código_Inflightor', 'Produto\\Receita',
        'Classe', 'Ciclo', '%', 'Quantidade', 'UN', 'Custo'
    ]

    for filename, content in uploaded.items():
        print(f'\n--- Processando Relatório: {filename} ---')
        df_raw = pd.read_excel(io.BytesIO(content), header=None)

        rows_list = []
        current_cia = ""
        current_classe_meta = ""
        current_ciclo = ""
        current_base_custo = ""
        current_cardapio = ""
        current_resumo = ""
        current_percentual = "" # Variável para armazenar o % extraído do cardápio

        for i in range(len(df_raw)):
            col_a = str(df_raw.iloc[i, 0]).strip().upper() if pd.notnull(df_raw.iloc[i, 0]) else ""

            # 1. Busca Metadados Gerais dinâmicos
            for c in range(len(df_raw.columns)):
                cell_val = str(df_raw.iloc[i, c]).strip().upper() if pd.notnull(df_raw.iloc[i, c]) else ""
                if "COMPANHIA" in cell_val:
                    current_cia = str(df_raw.iloc[i, c+2]).strip() if c+2 < len(df_raw.columns) else ""
                elif "CLASSE" in cell_val and "CLASSE:" not in cell_val and col_a not in ["P", "R", "IT:"]:
                    current_classe_meta = str(df_raw.iloc[i, c+1]).strip() if c+1 < len(df_raw.columns) else ""
                elif "CICLO" in cell_val:
                    current_ciclo = str(df_raw.iloc[i, 7]).strip() if pd.notnull(df_raw.iloc[i, 7]) else ""
                elif cell_val == "CUSTO" and col_a not in ["P", "R", "IT:"]:
                    current_base_custo = str(df_raw.iloc[i, c+1]).strip() if c+1 < len(df_raw.columns) else ""

            # 2. Identifica Novo Cardápio (IT:)
            if col_a == "IT:":
                full_name = str(df_raw.iloc[i, 2]).strip() if pd.notnull(df_raw.iloc[i, 2]) else ""
                current_cardapio = full_name
                current_resumo = re.split(r'\s*\(', full_name)[0].strip()

                # Extrai a porcentagem de dentro dos parênteses do Cardápio (ex: "COCO (50%)" -> "50%")
                match = re.search(r'\(([^)]*%[^)]*)\)', full_name)
                current_percentual = match.group(1).strip() if match else ""

                # Adiciona a linha do IT: como Produto\Receita também
                row_it = {
                    'Cia': current_cia, 'Cardapio': current_cardapio, 'Resumo_Cardapio': current_resumo,
                    'Class': '', 'Produto\\Receita': full_name, 'Classe': current_classe_meta,
                    'Ciclo': current_ciclo, 'Base de Custo': current_base_custo, '%': current_percentual
                }
                rows_list.append(row_it)
                continue

            # 3. Identifica Itens (P ou R)
            if col_a in ["P", "R"]:
                row_data = {
                    'Cia': current_cia,
                    'Cardapio': current_cardapio,
                    'Resumo_Cardapio': current_resumo,
                    'Class': col_a,
                    'Classe': current_classe_meta,
                    'Ciclo': current_ciclo,
                    'Base de Custo': current_base_custo,
                    'Código_Inflightor': df_raw.iloc[i, 1],
                    'Produto\\Receita': df_raw.iloc[i, 2],
                    'Quantidade': df_raw.iloc[i, 7],
                    'UN': df_raw.iloc[i, 8],
                    'Custo': df_raw.iloc[i, 9],
                    '%': current_percentual # Atribui a porcentagem extraída do cardápio correspondente
                }
                rows_list.append(row_data)

        df_final = pd.DataFrame(rows_list)

        # Limpeza da coluna Produto\Receita: remove tudo após o primeiro (
        if 'Produto\\Receita' in df_final.columns:
            df_final['Produto\\Receita'] = df_final['Produto\\Receita'].astype(str).str.split('(').str[0].str.strip()

        for col in colunas_modelo:
            if col not in df_final.columns: df_final[col] = ""

        df_final = df_final[colunas_modelo]
        output_name = f'VERTICALIZADO_{filename}'
        df_final.to_excel(output_name, index=False)
        files.download(output_name)

processar_relatorio_listagem()

Saving LISTAGEM 20.03.26 LATAM GRU.xlsx to LISTAGEM 20.03.26 LATAM GRU (1).xlsx

--- Processando Relatório: LISTAGEM 20.03.26 LATAM GRU (1).xlsx ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>